In [0]:
spark.conf.set(
    "fs.azure.account.key.airlineanalyticsprojsrc.dfs.core.windows.net",
    "storage acc key"
)

In [0]:
from pyspark.sql.functions import col, length, to_date, when, avg, count

# 1. Sabse pehle main flights history read karna
raw_flights_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("abfss://airline-flights-data@airlineanalyticsprojsrc.dfs.core.windows.net/bronze/flights_historical.csv")

# 2. Baaki saari lookup files ko Bronze layer me Delta Tables banana
lookup_tables = ["airlines_lookup", "airports_lookup", "flight_data_2024_data_dictionary"]

for table in lookup_tables:
    df_lookup = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .load(f"abfss://airline-flights-data@airlineanalyticsprojsrc.dfs.core.windows.net/bronze/{table}.csv")
    
    df_lookup.write.format("delta").mode("overwrite") \
        .save(f"abfss://airline-flights-data@airlineanalyticsprojsrc.dfs.core.windows.net/bronze/bronze_{table}")

print("Bronze Layer completely ready with tables! 🔥")

In [0]:
print("Applying 10 Advanced Data Quality Checks on Flights Data...")

# 1. Main Flights Silver Table (With 10 Advanced Constraints)
cleaned_flights_df = raw_flights_df \
    .dropDuplicates() \
    .filter(col("month").between(1, 12)) \
    .filter(col("day_of_month").between(1, 31)) \
    .filter(col("distance") > 0) \
    .filter(to_date(col("fl_date"), "yyyy-MM-dd").isNotNull()) \
    .filter(length(col("op_unique_carrier")) == 2) \
    .filter((length(col("origin")) == 3) & (length(col("dest")) == 3)) \
    .filter(col("cancelled").isin(0, 1)) \
    .filter(col("diverted").isin(0, 1)) \
    .withColumn("weather_delay", when(col("weather_delay") < 0, 0).otherwise(col("weather_delay"))) \
    .withColumn("carrier_delay", when(col("carrier_delay") < 0, 0).otherwise(col("carrier_delay")))

cleaned_flights_df.write.format("delta").mode("overwrite").save("abfss://airline-flights-data@airlineanalyticsprojsrc.dfs.core.windows.net/silver/silver_flights")

# 2. Airlines Lookup Silver
silver_airlines = spark.read.format("delta").load("abfss://airline-flights-data@airlineanalyticsprojsrc.dfs.core.windows.net/bronze/bronze_airlines_lookup") \
    .filter(col("op_unique_carrier").isNotNull() & (length(col("op_unique_carrier")) == 2))
silver_airlines.write.format("delta").mode("overwrite").save("abfss://airline-flights-data@airlineanalyticsprojsrc.dfs.core.windows.net/silver/silver_airlines")

# 3. Airports Lookup Silver
silver_airports = spark.read.format("delta").load("abfss://airline-flights-data@airlineanalyticsprojsrc.dfs.core.windows.net/bronze/bronze_airports_lookup") \
    .filter(col("airport_code").isNotNull() & (length(col("airport_code")) == 3))
silver_airports.write.format("delta").mode("overwrite").save("abfss://airline-flights-data@airlineanalyticsprojsrc.dfs.core.windows.net/silver/silver_airports")

# 4. Data Dictionary Silver
silver_dict = spark.read.format("delta").load("abfss://airline-flights-data@airlineanalyticsprojsrc.dfs.core.windows.net/bronze/bronze_flight_data_2024_data_dictionary")
silver_dict.write.format("delta").mode("overwrite").save("abfss://airline-flights-data@airlineanalyticsprojsrc.dfs.core.windows.net/silver/silver_data_dictionary")

print("Silver Layer perfectly upgraded with 10 Advanced Data Quality Rules! 💎")

In [0]:
display(spark.sql("DESCRIBE HISTORY delta.`abfss://airline-flights-data@airlineanalyticsprojsrc.dfs.core.windows.net/silver/silver_flights`"))